In [ ]:
!pip install trl

In [ ]:
!pip install -U torchao

In [ ]:
dataset = {
  "messages": [
    {"role": "system", "content": "Bạn là trợ lý dịch thuật."},
    {"role": "user", "content": "Dich: Hom nay troi dep."},
    {"role": "assistant", "content": "The weather is nice today."}
  ]
}

In [ ]:
from transformers import AutoTokenizer

# Load the tokenizer that ships with the target model
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")

messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is LoRA?"},
]

# Render the messages into the exact prompt string the model expects
prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,            # return a string, not token ids
    add_generation_prompt=True # append the assistant turn opener for inference
)
print(prompt)

In [ ]:
import gc, torch
print(f"VRAM đang dùng: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"VRAM reserved: {torch.cuda.memory_reserved()/1e9:.2f} GB")

In [ ]:
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
from transformers import AutoModelForCausalLM, AutoTokenizer
import datetime
import peft

model_id = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)

lora_config = peft.LoraConfig(
    r=8,
    lora_alpha=8,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
)

lora_model = peft.get_peft_model(model, lora_config)
lora_model.print_trainable_parameters()

# Gradient Checkpointing: Tiết kiệm bộ nhớ bằng cách chỉ lưu trữ một phần các kích hoạt (activations) 
# và tính toán lại phần còn lại khi cần trong lượt truyền ngược. Phương pháp này giúp huấn luyện các mô hình lớn nhưng làm chậm tốc độ huấn luyện khoảng 20%


# Dataset must contain a "messages" column of role/content dicts
dataset = load_dataset("HuggingFaceH4/ultrachat_200k", split="train_sft")
dataset = dataset.remove_columns(
    [col for col in dataset.column_names if col != "messages"]
)
dataset = dataset.select(range(2000)) 

config = SFTConfig(
    output_dir="./sft-out",
    max_length=1024,          # truncate long conversations
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    gradient_checkpointing=False,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    num_train_epochs=1,
    fp16=True,
)


trainer = SFTTrainer(
    model=lora_model,
    args=config,
    train_dataset=dataset,
    processing_class=tokenizer,  # TRL applies the chat template automatically
)
start = datetime.datetime.now()
trainer.train()
end = datetime.datetime.now()
print("Training time:", end - start)